In [ ]:
import sys, os, io, json, shutil, tempfile, unittest
WS = r'd:\OneDrive\Q-py - Copy'
if WS not in sys.path:
    sys.path.insert(0, WS)
os.chdir(WS)
import app
print('IMPORT OK')
print('HAS_SCAN_LIBS', app.HAS_SCAN_LIBS, '| HAS_TESSERACT', app.HAS_TESSERACT, '| HAS_PYZBAR', app.HAS_PYZBAR, '| HAS_DOCX', app.HAS_DOCX, '| HAS_OPENPYXL', app.HAS_OPENPYXL, '| HAS_QRCODE', app.HAS_QRCODE)
print('tesseract:', app.tesseract_cmd())
_tmp = tempfile.mkdtemp(prefix='exam_test_')
app.DB_PATH = os.path.join(_tmp, 'exam_bank.db')
shutil.copyfile(os.path.join(WS, 'exam_bank.db'), app.DB_PATH)
app.BACKUP_DIR = os.path.join(_tmp, 'backups')
app.BACKUP_PATH = os.path.join(app.BACKUP_DIR, 'backup_exam_bank.db')
print('Temp DB ready:', app.DB_PATH)

In [ ]:
suite = unittest.TestLoader().loadTestsFromName('test_exam_workflow')
result = unittest.TextTestRunner(verbosity=2).run(suite)
print('UNIT TESTS:', 'OK' if result.wasSuccessful() else 'FAILED')

In [ ]:
conn = app.get_db()
for r in conn.execute('SELECT id, subject, cohort, key_transferred, print_confirmed, grading_json, model_a_json FROM exam_issues'):
    d = dict(r)
    blocks = json.loads(d['model_a_json'])
    print('issue:', d['subject'], '| cohort', d['cohort'], '| key_transferred =', d['key_transferred'], '| blocks =', len(blocks), '| grading =', (d['grading_json'] or '')[:80])
rows = conn.execute('SELECT issue_id, company, serial_number, model, status, military_number FROM exam_booklets ORDER BY issue_id, company, serial_number').fetchall()
print('booklets total:', len(rows))
for r in rows[:40]:
    print(dict(r))
conn.close()

In [ ]:
PDF = os.path.join(WS, 'Scan_20261017 - Copy (5).pdf')
print('exists:', os.path.exists(PDF), '| size KB:', os.path.getsize(PDF) // 1024 if os.path.exists(PDF) else 0)
data = open(PDF, 'rb').read()
detector = app.cv2.QRCodeDetector()
conn = app.get_db()
secret = app.qr_signing_secret(conn)
conn.commit()
conn.close()
for page_no, image in app.iter_scan_pages('scan.pdf', data):
    code = app.read_qr_from_image(image, detector)
    payload = app.decode_exam_qr(code, secret) if code else None
    bubbles = app.detect_page_bubbles(image)
    marked = sum(1 for b in bubbles if b['marked'])
    uncertain = sum(1 for b in bubbles if b['uncertain'])
    print('page', page_no, '| size', image.shape[1], 'x', image.shape[0], '| QR:', 'YES' if payload else 'NO', '| bubbles:', len(bubbles), '| marked:', marked, '| uncertain:', uncertain)
    if payload:
        print('   payload:', payload)

In [ ]:
client = app.app.test_client()
resp = client.post('/api/exams/scan', data={'files': [(io.BytesIO(data), 'Scan_20261017.pdf')]}, content_type='multipart/form-data')
print('HTTP', resp.status_code)
out = resp.get_json()
print('engines:', out.get('engines'))
print('booklets analyzed:', len(out.get('booklets', [])))
for b in out.get('booklets', []):
    row = app.get_db().execute('SELECT responses_json FROM exam_booklets WHERE qr_token = ?', (b['qr_token'],)).fetchone()
    correct = sum(1 for v in json.loads(row[0] or '{}').values() if v.get('correct'))
    print('booklet serial', b['serial_number'], '| company', b['company'], '| model', b['model'], '| by', b['identified_by'], '| review items:', len(b['review']), '| auto-correct answers:', correct)
for p in out.get('page_problems', []):
    print('problem page', p['page'], ':', p['message'])